# MintDim TPU: train seed sweep

This notebook copies the current `mintdim-lab` repo to the Colab runtime SSD, installs the TPU JAX build, and trains seeds through `mintdim_lab.cli.commands.train` using `recipes/train/linear_equation_tiny_tpu.yaml`. Checkpoints, metrics, determinism logs, and the persistent JAX compilation cache are stored on Google Drive.


In [ ]:
# 1. Mount Google Drive
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# 2. Copy the repo and persistent JAX cache to the Colab runtime SSD
import os
import shutil
import sys
from pathlib import Path

from IPython.display import Markdown, display

status = display(Markdown("### Preparing runtime workspace..."), display_id=True)


def set_status(text: str) -> None:
    status.update(Markdown(text))


DRIVE_ROOT = Path("/content/drive/MyDrive")
DRIVE_REPO_CANDIDATES = [
    DRIVE_ROOT / "mintdim__lab",
    DRIVE_ROOT / "mintdim-lab",
    DRIVE_ROOT / "Colab Notebooks" / "mintdim__lab",
]
DRIVE_REPO = next((path for path in DRIVE_REPO_CANDIDATES if path.is_dir()), None)
if DRIVE_REPO is None:
    raise FileNotFoundError(
        "MintDim repo was not found on Drive. Update DRIVE_REPO_CANDIDATES "
        "to match the uploaded folder."
    )

COLAB_REPO = Path("/content/mintdim__lab")
set_status(f"### Copying repo to runtime SSD...  \n`{DRIVE_REPO}` -> `{COLAB_REPO}`")
if COLAB_REPO.exists():
    shutil.rmtree(COLAB_REPO)
shutil.copytree(
    DRIVE_REPO,
    COLAB_REPO,
    ignore=shutil.ignore_patterns(
        "__pycache__",
        "*.pyc",
        ".git",
        ".ipynb_checkpoints",
        ".pytest_cache",
        ".mypy_cache",
        ".ruff_cache",
        ".sixth",
        "node_modules",
        "experiments",
        "runs",
    ),
)

required_paths = [
    COLAB_REPO / "data_store/packed/linear_equation_unit96/unit_96/shard_000000.bin",
    COLAB_REPO / "data_store/raw/linear_equation/eval.jsonl",
    COLAB_REPO / "data_store/tokenizers/byte_bpe_512/tokenizer.json",
]
missing = [path for path in required_paths if not path.is_file() or path.stat().st_size == 0]
if missing:
    raise FileNotFoundError("Missing required repo artifacts: " + ", ".join(str(path) for path in missing))

os.chdir(COLAB_REPO)
for import_path in (COLAB_REPO, COLAB_REPO / "src"):
    path_text = str(import_path)
    if path_text not in sys.path:
        sys.path.insert(0, path_text)

# JAX must not be imported before cache/runtime environment variables are set.
if "jax" in sys.modules:
    raise RuntimeError(
        "JAX is already imported. Restart the runtime, then run this notebook "
        "from cell 1 so cache settings take effect."
    )

# Set the cache before importing JAX. Drive is durable; runtime SSD is active.
DRIVE_JAX_CACHE_DIR = DRIVE_ROOT / "mintdim__lab_jax_cache" / "tpu"
RUNTIME_JAX_CACHE_DIR = Path("/content/jax-cache/tpu")
DRIVE_JAX_CACHE_DIR.mkdir(parents=True, exist_ok=True)
if RUNTIME_JAX_CACHE_DIR.exists():
    shutil.rmtree(RUNTIME_JAX_CACHE_DIR)
if any(path.is_file() for path in DRIVE_JAX_CACHE_DIR.rglob("*")):
    shutil.copytree(DRIVE_JAX_CACHE_DIR, RUNTIME_JAX_CACHE_DIR)
else:
    RUNTIME_JAX_CACHE_DIR.mkdir(parents=True, exist_ok=True)

os.environ["MINTDIM_JAX_CACHE_DIR"] = str(RUNTIME_JAX_CACHE_DIR)
os.environ["JAX_COMPILATION_CACHE_DIR"] = str(RUNTIME_JAX_CACHE_DIR)
os.environ.setdefault("JAX_ENABLE_COMPILATION_CACHE", "true")
os.environ.setdefault("JAX_PERSISTENT_CACHE_MIN_COMPILE_TIME_SECS", "0")
os.environ.setdefault("JAX_PERSISTENT_CACHE_MIN_ENTRY_SIZE_BYTES", "0")


def sync_jax_cache_to_drive() -> None:
    DRIVE_JAX_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    for source in RUNTIME_JAX_CACHE_DIR.rglob("*"):
        if not source.is_file():
            continue
        target = DRIVE_JAX_CACHE_DIR / source.relative_to(RUNTIME_JAX_CACHE_DIR)
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.exists() or target.stat().st_size != source.stat().st_size:
            shutil.copy2(source, target)


set_status(
    f"### Runtime workspace ready  \n"
    f"Repo `{COLAB_REPO}`  \n"
    f"Training shard `{required_paths[0]}`  \n"
    f"Eval data `{required_paths[1]}`  \n"
    f"Tokenizer `{required_paths[2]}`  \n"
    f"Runtime JAX cache `{RUNTIME_JAX_CACHE_DIR}`"
)


In [ ]:
# 3. Install the TPU JAX build and runtime dependencies
import os
import subprocess
import sys
from pathlib import Path

set_status("### Removing stale JAX/CUDA packages...")
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "jax",
        "jaxlib",
        "jax-cuda12-plugin",
        "jax-cuda12-pjrt",
        "jax-cuda13-plugin",
        "jax-cuda13-pjrt",
    ],
    check=False,
)

set_status("### Installing JAX TPU build...")
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "jax[tpu]",
        "-f",
        "https://storage.googleapis.com/jax-releases/libtpu_releases.html",
    ],
    check=True,
)

set_status("### Installing MintDim lab runtime dependencies...")
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "mintdim>=0.1.22",
        "flax",
        "optax",
        "orbax-checkpoint",
        "PyYAML",
    ],
    check=True,
)

import jax  # noqa: E402
import mintdim_lab  # noqa: E402

tpu_devices = jax.devices("tpu")
if not tpu_devices:
    raise RuntimeError("No TPU devices detected. Select a TPU runtime in Colab first.")

set_status(
    f"### TPU environment ready  \n"
    f"JAX `{jax.__version__}`  \n"
    f"Backend `{jax.default_backend()}`  \n"
    f"TPU devices `{len(tpu_devices)}`  \n"
    f"mintdim_lab `{Path(mintdim_lab.__file__).resolve()}`"
)


In [ ]:
# 4. Train seeds through the current MintDim Lab train command
import gc
import json
from datetime import UTC, datetime

from mintdim_lab.cli.commands.train import main as train_main  # noqa: E402

SEED_START = 0
SEED_COUNT = 20
SEEDS = tuple(range(SEED_START, SEED_START + SEED_COUNT))
TRAINING_CONFIG = "recipes/train/linear_equation_tiny_tpu.yaml"
RESULTS_ROOT = DRIVE_ROOT / "mintdim__lab_tpu_seed_runs"
SKIP_FINISHED = True
ENABLE_DETERMINISM_LOG = True
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Seeds: {SEEDS}")
print(f"Training config: {TRAINING_CONFIG}")
print(f"Results root: {RESULTS_ROOT}")

for index, seed in enumerate(SEEDS, start=1):
    seed_dir = RESULTS_ROOT / f"seed_{seed:02d}"
    checkpoint_dir = seed_dir / "checkpoints"
    metrics_path = seed_dir / "train_metrics.jsonl"
    determinism_path = seed_dir / "determinism.jsonl"
    done_marker = seed_dir / "done.json"
    seed_dir.mkdir(parents=True, exist_ok=True)

    if SKIP_FINISHED and done_marker.is_file():
        print(f"[{index:02d}/{SEED_COUNT:02d}] seed={seed}: already complete, skipping")
        continue

    args = [
        "--repo",
        str(COLAB_REPO),
        "--training-config",
        TRAINING_CONFIG,
        "--seed",
        str(seed),
        "--checkpoint-dir",
        str(checkpoint_dir),
        "--jsonl",
        str(metrics_path),
        "--ui",
        "--log-every",
        "10",
    ]
    if ENABLE_DETERMINISM_LOG:
        args.extend(["--determinism-log", str(determinism_path)])

    print(f"[{index:02d}/{SEED_COUNT:02d}] seed={seed}: starting TPU train")
    print("mintdim_lab.cli.commands.train " + " ".join(args), flush=True)
    try:
        exit_code = train_main(args)
        if exit_code != 0:
            raise RuntimeError(f"Training CLI returned exit code {exit_code} for seed {seed}")
    finally:
        sync_jax_cache_to_drive()
        gc.collect()

    done_marker.write_text(
        json.dumps(
            {
                "seed": seed,
                "completed_at": datetime.now(UTC).isoformat(),
                "training_config": TRAINING_CONFIG,
                "checkpoint_dir": str(checkpoint_dir),
                "metrics_path": str(metrics_path),
                "determinism_log": str(determinism_path) if ENABLE_DETERMINISM_LOG else None,
            },
            indent=2,
        ),
        encoding="utf-8",
    )
    print(f"[{index:02d}/{SEED_COUNT:02d}] seed={seed}: complete", flush=True)

sync_jax_cache_to_drive()
print(f"All requested seeds complete: {SEEDS}")


In [ ]:
# 5. Evaluate the latest checkpoint for each completed seed
import gc
import json
import os
from pathlib import Path

from mintdim_lab.evaluator.run_evaluation import checkpoint_step, run_task  # noqa: E402

EVAL_CONFIG = COLAB_REPO / "recipes/evaluation/math_exact.yaml"
VOCAB_PATH = COLAB_REPO / "data_store/tokenizers/byte_bpe_512/tokenizer.json"
SKIP_EXISTING_EVAL = True


def latest_step_checkpoint(checkpoint_root: Path) -> Path:
    candidates = [
        path
        for path in checkpoint_root.glob("step_*")
        if path.is_dir() and checkpoint_step(path) is not None
    ]
    if not candidates:
        raise FileNotFoundError(f"No step_* checkpoint found in {checkpoint_root}")
    return max(candidates, key=lambda path: checkpoint_step(path) or -1)


os.chdir(COLAB_REPO)
eval_results = []

for index, seed in enumerate(SEEDS, start=1):
    seed_dir = RESULTS_ROOT / f"seed_{seed:02d}"
    done_marker = seed_dir / "done.json"
    checkpoint_root = seed_dir / "checkpoints"

    if not done_marker.is_file():
        print(f"[{index:02d}/{SEED_COUNT:02d}] seed={seed}: training is not complete, skipping")
        continue

    checkpoint = latest_step_checkpoint(checkpoint_root)
    json_path = checkpoint / "benchmark" / "math_exact.json"

    if SKIP_EXISTING_EVAL and json_path.is_file():
        payload = json.loads(json_path.read_text(encoding="utf-8"))["benchmark"]
        eval_results.append(payload)
        print(
            f"[{index:02d}/{SEED_COUNT:02d}] seed={seed}: existing eval "
            f"step={payload.get('step')} accuracy={payload.get('accuracy'):.4f}"
        )
        continue

    print(f"[{index:02d}/{SEED_COUNT:02d}] seed={seed}: evaluating `{checkpoint.name}`")
    result = run_task(
        EVAL_CONFIG,
        overrides={
            "checkpoint": checkpoint,
            "vocab_path": VOCAB_PATH,
            "json_output": json_path,
        },
    )
    eval_results.append(result)
    print(f"[{index:02d}/{SEED_COUNT:02d}] wrote `{json_path}`", flush=True)
    gc.collect()

print("\nEval summary")
for result in eval_results:
    print(
        f"seed={result.get('seed')} step={result.get('step')} "
        f"accuracy={result.get('accuracy'):.4f} "
        f"correct={result.get('correct')}/{result.get('total')}"
    )
